## Data Processing

In [ ]:
from pathlib import Path
from os.path import splitext
import numpy as np
import pandas as pd

PROCESSING_DIR = Path(".")
ABLATION_DIR = PROCESSING_DIR / "results/debloated"
BASELINE_DIR = PROCESSING_DIR / "results/baseline"

PRICE_PER_MS_PER_MB_PER_100K_REQ = 0.00000162109


def get_estimated_cost(mem, time):
    # mem in MB, time in ms
    if mem < 128:
        mem = 128
    return mem * PRICE_PER_MS_PER_MB_PER_100K_REQ * time


def filter_outliers(df: pd.DataFrame, column: str, threshold: float = 3.0):
    """
    Filters out outliers from a DataFrame based on the Z-score method.
    :param df: DataFrame containing the data.
    :param column: Column name to check for outliers.
    :param threshold: Z-score threshold to consider a value as an outlier.
    :return: DataFrame with outliers removed.
    """
    z_scores = np.abs((df[column] - df[column].mean()) / df[column].std())
    return df[z_scores < threshold]

# collect varying k result and generate into a pandas dataframe
def collect_data(app_name: str, scoring: str):

    key = ["e2e_latency", "mem_used", "billed_duration", "import_time"]

    if not (ABLATION_DIR / f"{app_name}_k20_{scoring}.csv").exists():
        raise FileNotFoundError(f"File {ABLATION_DIR}/{app_name}_k20_{scoring}.csv does not exist.")


    debloated_data = pd.read_csv(ABLATION_DIR / f"{app_name}_k20_{scoring}.csv")
    debloated_data["billed_duration"] /= 1000  # convert to seconds
    # calculate cost for each row
    debloated_data["cost"] = debloated_data.apply(
        lambda row: get_estimated_cost(row["mem_used"], row["billed_duration"]), axis=1
    )

    # filter outliers
    debloated_data = filter_outliers(debloated_data, "e2e_latency")

    return debloated_data

In [ ]:
def get_baseline_data(app_name: str):
    baseline_file = pd.read_csv(BASELINE_DIR / f"{app_name}.csv")
    baseline_file["billed_duration"] /= 1000
    baseline_data = {
        "mem_used": baseline_file["mem_used"].mean(),
        "e2e_latency": baseline_file["e2e_latency"].mean(),
        "billed_duration": baseline_file["billed_duration"].mean(),
        "import_time": baseline_file["import_time"].mean(),
    }
    baseline_data["cost"] = get_estimated_cost(
        baseline_data["mem_used"].mean(), baseline_data["billed_duration"].mean()
    )
    return baseline_data

## Plotting

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_boxplots(app_name: str):

    baseline_data = get_baseline_data(app_name)

    scoring = ["memory", "time", "cost", "random"]
    data = {}
    for s in scoring:
        debloated_df = collect_data(app_name, s)
        data[s] = {
            "e2e_latency": debloated_df["e2e_latency"],
            "mem_used": debloated_df["mem_used"],
            "cost": debloated_df["cost"],
        }

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    sns.set(style="whitegrid")
    for i, metric in enumerate(["e2e_latency", "mem_used", "cost"]):
        box_data = []
        labels = ["memory", "time", "combined", "random"]
        
        # Add debloated data
        for s in scoring:
            # Calculate improvement % over baseline for each metric
            baseline_value = baseline_data[metric]
            improvement = 100 * (baseline_value - data[s][metric].values) / baseline_value
            box_data.append(improvement)

        # Create boxplot
        sns.boxplot(data=box_data, ax=axes[i])
        axes[i].set_title(f"{metric.replace('_', ' ').title()} Improvement")
        axes[i].set_xticklabels(labels, rotation=45)
        axes[i].set_ylabel(metric.replace("_", " ").title())
        axes[i].set_xlabel("Scoring Method")
        # Set y-limits to min/max ±5
        all_vals = np.concatenate([arr for arr in box_data])
        ymin, ymax = all_vals.min(), all_vals.max()
        axes[i].set_ylim(ymin - 5, ymax + 5)



    plt.suptitle(f"Improvements for {app_name}")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(PROCESSING_DIR / f"figures/boxplot-{app_name}.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
plot_boxplots("dna-visualization")

In [ ]:
plot_boxplots("lightgbm")

In [ ]:
plot_boxplots("spacy")